# 🎿 Skiing CNN + DQN Agent (Speed Only)
Train a reinforcement learning agent with the sole purpose of reaching the bottom of the hill as fast as possible. Gates are ignored entirely.

In [ ]:
# Install dependencies
!pip install gymnasium[atari] stable-baselines3[extra] ale-py opencv-python

In [ ]:
# Environment setup
# We build the env manually instead of using AtariWrapper so we have
# full control over each wrapper. This avoids wrappers that aren't
# useful for a pure speed goal (e.g. NoopResetEnv, EpisodicLifeEnv).
import gymnasium as gym
import ale_py
import numpy as np

from stable_baselines3.common.atari_wrappers import MaxAndSkipEnv, WarpFrame
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

gym.register_envs(ale_py)

def make_env():
    env = gym.make("ALE/Skiing-v5")
    env = MaxAndSkipEnv(env, skip=4)  # Act every 4 frames for faster decisions
    env = WarpFrame(env)              # Resize to 84x84 grayscale for CNN input
    return env

env = DummyVecEnv([make_env])
env = VecFrameStack(env, n_stack=4)  # Stack 4 frames so the model can detect motion

In [ ]:
# Create CNN + DQN model
from stable_baselines3 import DQN

model = DQN(
    "CnnPolicy",
    env,
    verbose=1,
    learning_rate=1e-4,
    buffer_size=100000,
    learning_starts=10000,
    batch_size=32,
    tau=1.0,
    # Increased from 0.99 — makes the agent weight the final
    # time reward more heavily, which is the entire goal here.
    gamma=0.999,
    train_freq=4,
    target_update_interval=1000,
    # Reduced exploration — the task is simpler (just go fast)
    # so the agent doesn't need to explore as long.
    exploration_fraction=0.05,
    exploration_final_eps=0.005,
    device="auto",  # Will use CUDA GPU if available, otherwise CPU
)

In [ ]:
# Train the agent
# 1-2M steps is enough for a speed-only goal on a decent GPU.
# The task is simpler than gate-following so it converges faster.
model.learn(total_timesteps=2_000_000)
model.save("skiing_cnn_dqn_speedonly")

In [ ]:
# Run the trained agent
def make_render_env():
    env = gym.make("ALE/Skiing-v5", render_mode="human")
    env = MaxAndSkipEnv(env, skip=4)
    env = WarpFrame(env)
    return env

eval_env = DummyVecEnv([make_render_env])
eval_env = VecFrameStack(eval_env, n_stack=4)

model = DQN.load("skiing_cnn_dqn_speedonly", env=eval_env)

obs = eval_env.reset()

n_episodes = 5
episodes_done = 0

while episodes_done < n_episodes:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, info = eval_env.step(action)

    if done[0]:
        episodes_done += 1
        print(f"Episode {episodes_done} finished.")
        obs = eval_env.reset()

eval_env.close()